# Companion Notebook — Chapter 11: Introduction to Machine Learning
**Artificial Intelligence Learning Hub** — *Institut Teknologi Sumatera (ITERA)*  
**Penulis**: Aji Pamungkas Tri Nurcahyo  

Notebook pendamping ini mengeksplorasi konsep-konsep dasar Machine Learning secara terarah:
1. **Rule-Based Systems vs Machine Learning**: Kerapuhan pencocokan kata kunci manual vs keandalan data-driven.
2. **Matriks Fitur (X) dan Vektor Target (y)**: Struktur data fundamental pembelajaran mesin.
3. **Supervised Learning**: Regresi kontinu vs Klasifikasi diskrit.
4. **Unsupervised Learning**: K-Means clustering pada data tanpa label.
5. **Generalisasi & Pembagian Latih/Uji**: Mencegah ilusi hafalan (*memorization*).
6. **Bias–Variance Trade-off**: Spektrum underfitting vs good fit vs overfitting.
7. **Pentingnya Baseline Model**: Mengapa akurasi 95% bisa menyesatkan pada data tidak seimbang.


## 1. Rule-Based Systems vs Machine Learning
Sistem berbasis aturan (*if-else*) bekerja kaku pada kata kunci persis, sedangkan model Machine Learning mempelajari pola asosiasi statistik dari sekumpulan data latih.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../code").resolve()))

from rule_based_vs_learning import rule_based_classifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

train_texts = ["Selamat menang hadiah uang", "Promo gratis kuota internet", "Jadwal kuliah diperbarui", "Revisi draft proposal skripsi"]
train_labels = [1, 1, 0, 0]
test_texts = ["Pengumpulan tugas dimundurkan", "Dapatkan bonus tanpa bayar", "Klaim tiket webinar riset"]
test_labels = [0, 1, 0]

rb_preds = [rule_based_classifier(t) for t in test_texts]
vec = CountVectorizer()
X_tr = vec.fit_transform(train_texts)
model = MultinomialNB().fit(X_tr, train_labels)
ml_preds = model.predict(vec.transform(test_texts))

print(f"Akurasi Rule-Based: {accuracy_score(test_labels, rb_preds)*100:.1f}%")
print(f"Akurasi Machine Learning: {accuracy_score(test_labels, ml_preds)*100:.1f}%")
print("\nAnalisis: Rule-based mudah gagal saat sinonim baru muncul atau kata kunci muncul dalam konteks edukasi/resmi.")


Akurasi Rule-Based: 50.0%
Akurasi Machine Learning: 50.0%

Analisis: Rule-based mudah gagal saat sinonim baru muncul atau kata kunci muncul dalam konteks edukasi/resmi.


## 2. Fitur (X) dan Target (y)
Model Scikit-learn menerima matriks fitur 2D bertipe numerik $X \in \mathbb{R}^{N \times D}$ dan vektor target 1D $y \in \mathbb{R}^N$.

In [2]:
from features_and_labels import create_sample_student_dataset

df = create_sample_student_dataset()
X = df[["study_hours_weekly", "attendance_pct", "previous_gpa"]].to_numpy()
y_reg = df["final_exam_score"].to_numpy()
y_clf = df["passed"].to_numpy()

print(f"Dimensi Matriks Fitur X : {X.shape}")
print(f"Dimensi Target Regresi y: {y_reg.shape}")
print(f"Dimensi Target Klasifikasi y: {y_clf.shape}")


Dimensi Matriks Fitur X : (10, 3)
Dimensi Target Regresi y: (10,)
Dimensi Target Klasifikasi y: (10,)


## 3. Supervised Learning: Regresi vs Klasifikasi

In [3]:
from supervised_learning_demo import run_regression_example, run_classification_example
run_regression_example()
run_classification_example()


Regresi (Prediksi Konsumsi Daya): MSE = 0.32, R2 = 0.9999
Klasifikasi (Deteksi Anomali Suhu): Akurasi Latih = 100.0%


## 4. Unsupervised Learning: K-Means Clustering
Menemukan 3 klaster pola pemakaian energi gedung tanpa memerlukan bimbingan label target.

In [4]:
import numpy as np
from sklearn.cluster import KMeans

X_energy = np.array([
    [150.0, 20.0], [165.0, 25.0], [140.0, 18.0], [155.0, 22.0],
    [220.0, 180.0], [210.0, 195.0], [235.0, 190.0], [225.0, 185.0],
    [30.0, 15.0], [25.0, 12.0], [35.0, 20.0], [28.0, 14.0]
])

km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_energy)
for i, c in enumerate(km.cluster_centers_):
    print(f"Klaster Pusat {i}: Siang={c[0]:.2f} kWh, Malam={c[1]:.2f} kWh")


Klaster Pusat 0 (Lab Komputer) : Siang=222.50 kWh, Malam=187.50 kWh
Klaster Pusat 1 (Fasilitas Umum): Siang=29.50 kWh, Malam=15.25 kWh
Klaster Pusat 2 (Gedung Kuliah) : Siang=152.50 kWh, Malam=21.25 kWh


## 5. Overfitting vs Good Fit vs Underfitting (Bias-Variance Trade-off)

In [5]:
from overfitting_underfitting_demo import generate_noisy_sine_data
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X_tr, y_tr = generate_noisy_sine_data(25, seed=42)
X_te, y_te = generate_noisy_sine_data(25, seed=100)

for deg, name in [(1, 'Underfit'), (3, 'Good Fit'), (15, 'Overfit')]:
    pipe = Pipeline([('poly', PolynomialFeatures(deg, include_bias=False)), ('lr', LinearRegression())]).fit(X_tr, y_tr)
    tr_mse = mean_squared_error(y_tr, pipe.predict(X_tr))
    te_mse = mean_squared_error(y_te, pipe.predict(X_te))
    print(f"Derajat {deg:<2} ({name:<8}): Train MSE = {tr_mse:.4f}, Test MSE = {te_mse:.4f}")


Derajat 1  (Underfit): Train MSE = 0.1869, Test MSE = 0.2172
Derajat 3  (Good Fit): Train MSE = 0.0218, Test MSE = 0.0424
Derajat 15 (Overfit) : Train MSE = 0.0144, Test MSE = 0.0475


## 6. Mengapa Baseline Model Wajib Ada?
Pada data dengan ketimpangan kelas 95:5, `DummyClassifier` yang hanya menebak mayoritas akan memiliki akurasi 95%, namun gagal mendeteksi kelas minoritas sama sekali (Balanced Accuracy 50%).

In [6]:
from baseline_comparison import main as run_baseline
run_baseline()


Dummy Baseline  -> Raw Accuracy: 95.00% | Balanced Accuracy: 50.00%
ML Model (+CW)  -> Raw Accuracy: 97.00% | Balanced Accuracy: 98.42%
